# 1. Importar librerías

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import locale
for loc in ('es_ES.UTF-8', 'Spanish_Chile.1252', 'Spanish'):
    try: locale.setlocale(locale.LC_TIME, loc); break
    except locale.Error: pass

# 2. Par?metros de entrada

Se mantienen `df1`, `df2`, `df3` y las etapas originales. `df1` y `df3` ahora se leen directamente desde las hojas del Excel; `df2` sigue siendo `SNDTGIS_ACQ`. Solo se modifican estas dos rutas.

In [2]:
CARPETA_ENTRADA = Path(r'D:\AMS_EXP\notebook')
ARCHIVO_EXCEL = next(CARPETA_ENTRADA.glob('DB Avance*.xlsx'), None)
ARCHIVO_COORDENADAS = Path(r'D:\AMS_EXP\notebook\SONDAJE_MLP\SNDTGIS_ACQ_02-06-26.csv')
ARCHIVO_RESULTADO = Path.cwd() / 'Planilla_Union_NORMALIZADA.xlsx'
EXPORTAR_RESULTADO = False
if ARCHIVO_EXCEL is None or not ARCHIVO_EXCEL.is_file(): raise FileNotFoundError('No se encontro DB Avance*.xlsx')
if not ARCHIVO_COORDENADAS.is_file(): raise FileNotFoundError(ARCHIVO_COORDENADAS)

In [3]:
# df1: lectura directa de CONSOLIDADO_PROGRAMA.
df1 = pd.read_excel(ARCHIVO_EXCEL, sheet_name='CONSOLIDADO_PROGRAMA', header=3, engine='openpyxl')
df1 = df1.rename(columns={'Sondaje': 'NRO_SON'})
df1 = df1[df1['NRO_SON'].notna()].copy()

D:\Env\arcgis-pro\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [4]:
# df2: coordenadas externas ACQ, como en el notebook original.
df2 = pd.read_csv(ARCHIVO_COORDENADAS, sep=';', encoding='latin1')
df2 = df2[df2['NRO_SON'].notna()].copy()

In [5]:
# df3: lectura directa de AVANCE MUESTRERA.
df3 = pd.read_excel(ARCHIVO_EXCEL, sheet_name='AVANCE MUESTRERA', header=1, engine='openpyxl')
df3 = df3.rename(columns={'Sondaje': 'NRO_SON'})
df3 = df3[df3['NRO_SON'].notna()].copy()

D:\Env\arcgis-pro\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
D:\Env\arcgis-pro\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


# 3. Revisión datasets

In [6]:
print('df1 = CONSOLIDADO_PROGRAMA del Excel')
df1

df1 = CONSOLIDADO_PROGRAMA del Excel


,Unnamed: 0,ID,Programa,Tipo de Sondaje,NRO_SON,Sector,Este,Norte,Cota,Azimut,...,Fecha Termino,Por Perforar (m),Avance Actual (m),Mts. Faltantes,%Avance,Estatus Perforación (m),Largo Final (m),Certificado Collar,Fecha Medición de Trayectoria,Observación
0,NaN,EVU_58,EVU,DDH-HQ3,DDH3958,Mirador Parapeto,59858.0,91097.0,3567.0,270.0,...,2024-09-23,486,486.00,0.00,100.000000,Finalizado,486.00,NaN,NaN,NaN
1,NaN,EVU_55,EVU,DDH-HQ3,DDH3959,Mirador Parapeto,59851.0,91050.0,3568.0,270.0,...,2024-10-12,862,862.00,0.00,100.000000,Finalizado,862.00,NaN,NaN,NaN
2,NaN,EVU_62,EVU,DDH-HQ3,DDH3961,Mirador Parapeto,59844.0,91207.0,3560.0,267.0,...,2024-09-29,502,502.00,0.00,100.000000,Finalizado,502.00,NaN,NaN,NaN
3,NaN,EVU_103,EVU,DDH-HQ3,DDH3962,Mirador Parapeto,59800.0,91745.0,3534.0,298.0,...,2024-10-03,294,294.00,0.00,100.000000,Finalizado,294.00,NaN,NaN,NaN
4,NaN,EVU_54,EVU,DDH-HQ3,DDH3963,Mirador Parapeto,59889.0,91000.0,3570.0,270.0,...,2024-10-18,490,422.65,67.35,86.255102,Finalizado,422.65,NaN,NaN,"Quedan en fondo barril mas switch 0,9 mts de b..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,NaN,EVU143,EVU,DDH-HQ3,DDH4177,F12 Sector Norte Banco 3665,59251.0,92222.0,3669.0,144.0,...,NaT,600,171.80,428.20,28.633333,En Avance,NaN,NaN,NaN,NaN
129,NaN,TIG_04B,TIGRESA,DDH-HQ3,DDH4176,Fase 12 Sector Hito Banco 3598,60166.0,92670.0,3598.0,270.0,...,NaT,800,140.35,659.65,17.543750,En Avance,NaN,NaN,NaN,NaN
130,NaN,REC_PZVZ_44,HIDROGEOLOGÍA,DDH-HQ3,DDH4173,Fase 11,58358.0,91112.0,3295.0,45.0,...,NaT,400,22.65,377.35,5.662500,En Avance,NaN,NaN,NaN,NaN
131,NaN,REC26_23,GEOTÉCNICO,DDH-HQ3,DDH4178,F12 BANCO 3500,59092.0,91748.0,3500.0,152.0,...,NaT,370,0.00,370.00,0.000000,En Avance,NaN,NaN,NaN,NaN


In [7]:
print('df2 = SNDTGIS_ACQ')
df2

df2 = SNDTGIS_ACQ


,NUM,NRO_SON,NRO_RECOM,COD_PROYECTO,NOMBRE_PROYECTO,NRO_CAMAPANA,DES_PAIS,DES_DISTRITO,DES_CAMPANA,ANO_CAMPANA,...,AZIMUTH,INCLINA,LARGO,ESTE,NORTE,COTA,ESTE_REC,NORTE_REC,COTA_REC,SOPORTE
0,1,DDH1,NaN,10,MLP,41,CHILE,NaN,ANACONDA MINERALS 1980-1982,1980.0,...,56.15,-34.9,450.67,59158.32,90441.6,3167.12,59158.32,90441.6,3167.12,2
1,2,DDH2,NaN,10,MLP,41,CHILE,NaN,ANACONDA MINERALS 1980-1982,1980.0,...,269.8,-34.6,465.40,58909.6,90315.2,3177.35,58909.6,90315.2,3177.35,2
2,3,DDH3,NaN,10,MLP,41,CHILE,NaN,ANACONDA MINERALS 1980-1982,1980.0,...,57.77,-42.8,352.35,59352.31,90012.07,3149.19,59352.31,90012.07,3149.19,2
3,4,DDH4,NaN,10,MLP,41,CHILE,NaN,ANACONDA MINERALS 1980-1982,1980.0,...,312.48,-39.0,457.05,59230.94,91116.98,3448.89,59230.94,91116.98,3448.89,2
4,5,DDH5,NaN,10,MLP,41,CHILE,NaN,ANACONDA MINERALS 1980-1982,1980.0,...,0,-90.0,286.50,59521.76,91109.82,3493.7,59521.76,91109.82,3493.7,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3145,3146,DDHNU110,NaN,10,MLP,40,CHILE,NaN,UNITED NATIONS 1969-1971,1969.0,...,0,-90.0,250.24,59017.99,90516.69,3215.04,59017.99,90516.69,3215.04,2
3146,3147,DDHNU111,NaN,10,MLP,40,CHILE,NaN,UNITED NATIONS 1969-1971,1969.0,...,0,-90.0,155.45,59319.86,90030.45,3144.62,59319.86,90030.45,3144.62,2
3147,3148,REC_GM-23,REC_GM-23,10,MLP,808,CHILE,NaN,AÑOS 2023 - 2024,2023.0,...,NaN,NaN,0.00,NaN,NaN,NaN,59443,89968,2914,2
3148,3149,DDH3018_RET,NaN,10,MLP,766,CHILE,NaN,AÑOS 2014 - 2016 GEOTECNICO,2014.0,...,NaN,NaN,650.00,60263,89098,3481,60264,89098,3481,2


In [8]:
print('df3 = AVANCE MUESTRERA del Excel')
df3

df3 = AVANCE MUESTRERA del Excel


,Rec,Programa,NRO_SON,Estado de Pago,Sonda,Levantamiento de Collar,Azimut,Inclinación,Inicio,Término,...,Tricono,Fotografía,Corte,Desde3,Hasta3,Desde2,Hasta2,Unnamed: 30,Unnamed: 31,Unnamed: 32
0,EVU_58,EVU,DDH3958,Septiembre,CH1-130,NaN,270.0,-71,2024-09-09 00:00:00,2024-09-23,...,NaN,486.00,486.00,NaN,486.00,NaN,486.00,NaN,Etiquetas de fila,Suma de Tricono
1,EVU_55,EVU,DDH3959,Septiembre,CH1-125,NaN,270.0,-61,2024-09-09 00:00:00,2024-10-12,...,NaN,862.00,862.00,NaN,862.00,NaN,862.00,NaN,CATEGORIZACIÓN,NaN
2,EVU_62,EVU,DDH3961,Septiembre,CH1-122,NaN,267.0,-71,2024-09-16 00:00:00,2024-09-29,...,NaN,502.00,502.00,NaN,502.00,NaN,502.00,NaN,EVU,212.0
3,EVU_103,EVU,DDH3962,NaN,CH1-130,NaN,298.0,-66,2024-09-23 00:00:00,2024-10-03,...,NaN,294.00,294.00,NaN,294.00,NaN,294.00,NaN,GEOMETALÚRGICO,NaN
4,EVU_54,EVU,DDH3963,NaN,CH1-100,NaN,270.0,-69,2024-09-29 00:00:00,2024-10-18,...,NaN,422.65,422.65,NaN,422.65,NaN,422.65,NaN,GEOTÉCNICO,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,TIG_04B,TIGRESA,DDH4176,NaN,CH1-122,NaN,270.0,-55,2026-06-25 00:00:00,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,EVU_143,EVU,DDH4177,NaN,CH1-170,NaN,144.0,-47,2026-06-25 00:00:00,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
121,REC26_23,GEOTÉCNICO,DDH4178,NaN,CH1-99,NaN,152.0,-77,2026-07-02 00:00:00,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
122,GM26_71,GEOMETALÚRGICO,DDH4179,NaN,CH1-138,NaN,318.0,-71,2026-07-04 00:00:00,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Sin exportaci?n intermedia: df3 se revisa en memoria.

### Renombrar campos df2 Planilla ACQ

### Renombrar campos df2 Planilla ACQ

Se mantienen los campos `Q_*`.

In [10]:
df2 = df2.rename(columns={
    'NUM': 'Q_NUM' ,
    'NRO_RECOM': 'Q_NRO_RECOM',
    'COD_PROYECTO': 'Q_COD_PROYECTO',
    'NOMBRE_PROYECTO':'Q_NOMBRE_PROYECTO',
    'NRO_CAMAPANA': 'Q_NRO_CAMAPANA',
    'DES_PAIS': 'Q_DES_PAIS',
    'DES_DISTRITO': 'Q_DES_DISTRITO',
    'DES_CAMPANA': 'Q_DES_CAMPANA',
    'ANO_CAMPANA': 'Q_ANO_CAMPANA',
    'ANNO_SONDAJE': 'Q_ANNO_SONDAJE',
    'DES_TIPO_PERF': 'Q_DES_TIPO_PERF',
    'DES_ESTADO_SON': 'Q_DES_ESTADO_SON',
    'AZIMUTH': 'Q_AZIMUTH',
    'INCLINA': 'Q_INCLINA',
    'LARGO': 'Q_LARGO',
    'ESTE': 'Q_ESTE',
    'NORTE': 'Q_NORTE',
    'COTA': 'Q_COTA',
    'ESTE_REC': 'Q_ESTE_REC',
    'NORTE_REC': 'Q_NORTE_REC',
    'COTA_REC': 'Q_COTA_REC',
    'SOPORTE': 'Q_SOPORTE'
})

### Renombrar campos df3 Avance Muestrera

Se mantienen los campos `AV_*`.

In [11]:
df3= df3.rename(columns={
    'Rec':'AV_RECOMEND',
    'Programa':'AV_PROGRAMA',
    'Estado de Pago':'AV_ESTADO_PAGO',
    'Sonda':'AV_SONDA',
    'Levantamiento de Collar':'AV_LEVANT_COLLAR',
    'Azimut':'AV_AZIMUT',
    'Inclinación':'AV_INCLINACION',
    'Inicio':'AV_FCH_INI',
    'Término':'AV_FCH_TERM',
    'Estado':'AV_ESTADO_PERF',
    'Largo Programado':'AV_LARGO_PROGRAM',
    'Fondo Final':'AV_FONDO_FINAL',
    'Faltante':'AV_FALTANTE_PERF',
    '% Perforado':'AV_PCT_PERFORADO',
    'Medición':'AV_MEDICION',
    'Desde':'AV_MED_Desde',
    'Hasta':'AV_MED_Hasta',
    'Certificado Medición de Trayectoria':'AV_CERT_TRAYECTORIA',
    '%Recuperación de Sondajes':'AV_PCT_RECUPERACION',
    '% Rendimiento Sondajes':'AV_PCT_RENDIMIENTO',
    '% Desviación':'AV_PCT_DESVIACION',
    'Tricono':'AV_TRICONO',
    'Fotografía':'AV_M_FOTOGRAFIA',
    'Corte':'AV_M_CORTE',
    'Desde3':'AV_MAPEO_DESDE',
    'Hasta3':'AV_MAPEO_HASTA',
    'Desde2':'AV_PREPARACION_DESDE',
    'Hasta2':'AV_PREPARACION_HASTA',

})

In [12]:
print('df3 = Planilla_ACQ')
df3

df3 = Planilla_ACQ


,AV_RECOMEND,AV_PROGRAMA,NRO_SON,AV_ESTADO_PAGO,AV_SONDA,AV_LEVANT_COLLAR,AV_AZIMUT,AV_INCLINACION,AV_FCH_INI,AV_FCH_TERM,...,AV_TRICONO,AV_M_FOTOGRAFIA,AV_M_CORTE,AV_MAPEO_DESDE,AV_MAPEO_HASTA,AV_PREPARACION_DESDE,AV_PREPARACION_HASTA,Unnamed: 30,Unnamed: 31,Unnamed: 32
0,EVU_58,EVU,DDH3958,Septiembre,CH1-130,NaN,270.0,-71,2024-09-09 00:00:00,2024-09-23,...,NaN,486.00,486.00,NaN,486.00,NaN,486.00,NaN,Etiquetas de fila,Suma de Tricono
1,EVU_55,EVU,DDH3959,Septiembre,CH1-125,NaN,270.0,-61,2024-09-09 00:00:00,2024-10-12,...,NaN,862.00,862.00,NaN,862.00,NaN,862.00,NaN,CATEGORIZACIÓN,NaN
2,EVU_62,EVU,DDH3961,Septiembre,CH1-122,NaN,267.0,-71,2024-09-16 00:00:00,2024-09-29,...,NaN,502.00,502.00,NaN,502.00,NaN,502.00,NaN,EVU,212.0
3,EVU_103,EVU,DDH3962,NaN,CH1-130,NaN,298.0,-66,2024-09-23 00:00:00,2024-10-03,...,NaN,294.00,294.00,NaN,294.00,NaN,294.00,NaN,GEOMETALÚRGICO,NaN
4,EVU_54,EVU,DDH3963,NaN,CH1-100,NaN,270.0,-69,2024-09-29 00:00:00,2024-10-18,...,NaN,422.65,422.65,NaN,422.65,NaN,422.65,NaN,GEOTÉCNICO,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,TIG_04B,TIGRESA,DDH4176,NaN,CH1-122,NaN,270.0,-55,2026-06-25 00:00:00,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,EVU_143,EVU,DDH4177,NaN,CH1-170,NaN,144.0,-47,2026-06-25 00:00:00,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
121,REC26_23,GEOTÉCNICO,DDH4178,NaN,CH1-99,NaN,152.0,-77,2026-07-02 00:00:00,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
122,GM26_71,GEOMETALÚRGICO,DDH4179,NaN,CH1-138,NaN,318.0,-71,2026-07-04 00:00:00,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Sin exportaci?n intermedia: df3 renombrado se revisa en memoria.

### Revisión de atributos

In [14]:
print(df1.shape,'Planilla_MLP')
print(df1.columns.tolist())
print(df1.dtypes)

(133, 23) Planilla_MLP
['Unnamed: 0', 'ID', 'Programa', 'Tipo de Sondaje', 'NRO_SON', 'Sector', 'Este', 'Norte', 'Cota', 'Azimut', 'Inclinación', 'Largo (m)', 'Fecha Inicio', 'Fecha Termino', 'Por Perforar (m)', 'Avance Actual (m)', 'Mts. Faltantes', '%Avance', 'Estatus Perforación (m)', 'Largo Final (m)', 'Certificado Collar', 'Fecha Medición de Trayectoria', 'Observación']
Unnamed: 0                              float64
ID                                       object
Programa                                 object
Tipo de Sondaje                          object
NRO_SON                                  object
Sector                                   object
Este                                    float64
Norte                                   float64
Cota                                    float64
Azimut                                  float64
Inclinación                             float64
Largo (m)                                 int64
Fecha Inicio                     datetime64[ns

In [15]:
print(df2.shape,'Planilla_ACQ')
print(df2.columns.tolist())
print(df2.dtypes)

(3150, 23) Planilla_ACQ
['Q_NUM', 'NRO_SON', 'Q_NRO_RECOM', 'Q_COD_PROYECTO', 'Q_NOMBRE_PROYECTO', 'Q_NRO_CAMAPANA', 'Q_DES_PAIS', 'Q_DES_DISTRITO', 'Q_DES_CAMPANA', 'Q_ANO_CAMPANA', 'Q_ANNO_SONDAJE', 'Q_DES_TIPO_PERF', 'Q_DES_ESTADO_SON', 'Q_AZIMUTH', 'Q_INCLINA', 'Q_LARGO', 'Q_ESTE', 'Q_NORTE', 'Q_COTA', 'Q_ESTE_REC', 'Q_NORTE_REC', 'Q_COTA_REC', 'Q_SOPORTE']
Q_NUM                  int64
NRO_SON               object
Q_NRO_RECOM           object
Q_COD_PROYECTO         int64
Q_NOMBRE_PROYECTO     object
Q_NRO_CAMAPANA         int64
Q_DES_PAIS            object
Q_DES_DISTRITO       float64
Q_DES_CAMPANA         object
Q_ANO_CAMPANA        float64
Q_ANNO_SONDAJE         int64
Q_DES_TIPO_PERF       object
Q_DES_ESTADO_SON      object
Q_AZIMUTH             object
Q_INCLINA            float64
Q_LARGO              float64
Q_ESTE                object
Q_NORTE               object
Q_COTA                object
Q_ESTE_REC            object
Q_NORTE_REC           object
Q_COTA_REC            objec

### Revisión de valores NaN

In [16]:
print(df1.shape,'Planilla_MLP')
print(pd.isna(df1).sum())

(133, 23) Planilla_MLP
Unnamed: 0                       133
ID                                 0
Programa                           0
Tipo de Sondaje                    0
NRO_SON                            0
Sector                             0
Este                               0
Norte                              0
Cota                               0
Azimut                             0
Inclinación                        0
Largo (m)                          0
Fecha Inicio                       0
Fecha Termino                     14
Por Perforar (m)                   0
Avance Actual (m)                  0
Mts. Faltantes                     0
%Avance                            0
Estatus Perforación (m)            0
Largo Final (m)                   15
Certificado Collar               133
Fecha Medición de Trayectoria    133
Observación                      112
dtype: int64


In [17]:
print(df2.shape,'Planilla_ACQ')
print(pd.isna(df2).sum())

(3150, 23) Planilla_ACQ
Q_NUM                   0
NRO_SON                 0
Q_NRO_RECOM          2236
Q_COD_PROYECTO          0
Q_NOMBRE_PROYECTO       0
Q_NRO_CAMAPANA          0
Q_DES_PAIS              0
Q_DES_DISTRITO       3150
Q_DES_CAMPANA           0
Q_ANO_CAMPANA          76
Q_ANNO_SONDAJE          0
Q_DES_TIPO_PERF      2236
Q_DES_ESTADO_SON        0
Q_AZIMUTH              98
Q_INCLINA              98
Q_LARGO                 3
Q_ESTE                 26
Q_NORTE                26
Q_COTA                 26
Q_ESTE_REC              0
Q_NORTE_REC             0
Q_COTA_REC              0
Q_SOPORTE               0
dtype: int64


In [18]:
#print(df3.shape,'USUARIOS_AMSA_AREA')
#print(pd.isna(df3).sum())

### Revisión de valores únicos

In [19]:
#print(df1.shape,'USUARIOS_AMSA')
#print(df1.nunique())
#print('-----------------------------------------')

In [20]:
#print(df2.shape,'AREA_GRUPOS')
#print(df2.nunique())
#print('-----------------------------------------')

In [21]:
#print(df3.shape,'USUARIOS_AMSA_AREA')
#print(df3.nunique())
#print('-----------------------------------------')

# 5. Unión de Dataframe

#### Primera uni?n: consolidado del Excel + coordenadas SNDTGIS_ACQ

In [22]:
print('**** DF = Planila_MLP Planilla_ACQ ****')
DF = pd.merge(df1, df2, on=['NRO_SON'], how='outer', indicator=True)
print(DF.shape,'DF')
DF.head(5)

**** DF = Planila_MLP Planilla_ACQ ****
(3171, 46) DF


,Unnamed: 0,ID,Programa,Tipo de Sondaje,NRO_SON,Sector,Este,Norte,Cota,Azimut,...,Q_INCLINA,Q_LARGO,Q_ESTE,Q_NORTE,Q_COTA,Q_ESTE_REC,Q_NORTE_REC,Q_COTA_REC,Q_SOPORTE,_merge
0,NaN,EVU_58,EVU,DDH-HQ3,DDH3958,Mirador Parapeto,59858.0,91097.0,3567.0,270.0,...,-70.91,486.00,59855.78,91102.29,3560.35,59858,91097,3567,2.0,both
1,NaN,EVU_55,EVU,DDH-HQ3,DDH3959,Mirador Parapeto,59851.0,91050.0,3568.0,270.0,...,-60.73,862.00,59849.33,91053.27,3560.77,59851,91050,3568,2.0,both
2,NaN,EVU_62,EVU,DDH-HQ3,DDH3961,Mirador Parapeto,59844.0,91207.0,3560.0,267.0,...,-70.13,502.00,59842.29,91211.84,3552.9,59844,91207,3560,2.0,both
3,NaN,EVU_103,EVU,DDH-HQ3,DDH3962,Mirador Parapeto,59800.0,91745.0,3534.0,298.0,...,-66.25,294.00,59802.17,91741.71,3529.38,59800,91745,3534,2.0,both
4,NaN,EVU_54,EVU,DDH-HQ3,DDH3963,Mirador Parapeto,59889.0,91000.0,3570.0,270.0,...,-69.19,422.65,59886.78,91006.91,3561.87,59889,91000,3570,2.0,both


In [23]:
print('Columnas valores nulos')
print(pd.isna(DF).sum())

Columnas valores nulos
Unnamed: 0                       3171
ID                               3038
Programa                         3038
Tipo de Sondaje                  3038
NRO_SON                             0
Sector                           3038
Este                             3038
Norte                            3038
Cota                             3038
Azimut                           3038
Inclinación                      3038
Largo (m)                        3038
Fecha Inicio                     3038
Fecha Termino                    3052
Por Perforar (m)                 3038
Avance Actual (m)                3038
Mts. Faltantes                   3038
%Avance                          3038
Estatus Perforación (m)          3038
Largo Final (m)                  3053
Certificado Collar               3171
Fecha Medición de Trayectoria    3171
Observación                      3150
Q_NUM                              21
Q_NRO_RECOM                      2257
Q_COD_PROYECTO             

### Revisió de campos unión

In [24]:
print('**** DF = Union df1 y df2 ****')
print('=========================')
print(DF.columns.tolist())

**** DF = Union df1 y df2 ****
['Unnamed: 0', 'ID', 'Programa', 'Tipo de Sondaje', 'NRO_SON', 'Sector', 'Este', 'Norte', 'Cota', 'Azimut', 'Inclinación', 'Largo (m)', 'Fecha Inicio', 'Fecha Termino', 'Por Perforar (m)', 'Avance Actual (m)', 'Mts. Faltantes', '%Avance', 'Estatus Perforación (m)', 'Largo Final (m)', 'Certificado Collar', 'Fecha Medición de Trayectoria', 'Observación', 'Q_NUM', 'Q_NRO_RECOM', 'Q_COD_PROYECTO', 'Q_NOMBRE_PROYECTO', 'Q_NRO_CAMAPANA', 'Q_DES_PAIS', 'Q_DES_DISTRITO', 'Q_DES_CAMPANA', 'Q_ANO_CAMPANA', 'Q_ANNO_SONDAJE', 'Q_DES_TIPO_PERF', 'Q_DES_ESTADO_SON', 'Q_AZIMUTH', 'Q_INCLINA', 'Q_LARGO', 'Q_ESTE', 'Q_NORTE', 'Q_COTA', 'Q_ESTE_REC', 'Q_NORTE_REC', 'Q_COTA_REC', 'Q_SOPORTE', '_merge']


### Reemplazo coordenadas recomendadas por coordenadas BD AcQ 

In [25]:
mask = (
    (DF['_merge'] == 'both') &
    (DF['Q_ESTE'].notna()) &
    (DF['Q_NORTE'].notna()) &
    (DF['Q_COTA'].notna())
)

DF.loc[mask, 'Este'] = DF.loc[mask, 'Q_ESTE']
DF.loc[mask, 'Norte'] = DF.loc[mask, 'Q_NORTE']
DF.loc[mask, 'Cota'] = DF.loc[mask, 'Q_COTA']

### Validación rápida

In [26]:
print("Filas actualizadas:", mask.sum())

DF.loc[mask, [
    'NRO_SON',
    'Q_ESTE', 'Este',
    'Q_NORTE', 'Norte',
    'Q_COTA', 'Cota'
]].head()

Filas actualizadas: 96


,NRO_SON,Q_ESTE,Este,Q_NORTE,Norte,Q_COTA,Cota
0,DDH3958,59855.78,59855.78,91102.29,91102.29,3560.35,3560.35
1,DDH3959,59849.33,59849.33,91053.27,91053.27,3560.77,3560.77
2,DDH3961,59842.29,59842.29,91211.84,91211.84,3552.9,3552.9
3,DDH3962,59802.17,59802.17,91741.71,91741.71,3529.38,3529.38
4,DDH3963,59886.78,59886.78,91006.91,91006.91,3561.87,3561.87


### Deja solo both y left_only

In [27]:
DF = DF[DF['_merge'].isin(['both', 'left_only'])]

In [28]:
print(DF['_merge'].value_counts())
print(DF.shape)

_merge
both          112
left_only      21
right_only      0
Name: count, dtype: int64
(133, 46)


In [29]:
#mask = DF['_merge'] == 'both'

#DF.loc[mask, 'Este'] = DF.loc[mask, 'Este'] + 300000
#DF.loc[mask, 'Norte'] = DF.loc[mask, 'Norte'] + 6400000

### Conversi?n de coordenadas a valores num?ricos

In [30]:
DF['Este'] = pd.to_numeric(DF['Este'], errors='coerce')
DF['Norte'] = pd.to_numeric(DF['Norte'], errors='coerce')
DF['Cota'] = pd.to_numeric(DF['Cota'], errors='coerce')


### Aplicaci?n del falso Este y Norte

Se mantiene `+300000` y `+6400000`.

In [31]:
DF['Este'] = DF['Este'] + 300000
DF['Norte'] = DF['Norte'] + 6400000

In [32]:
print(DF.shape,'DF')
DF.head(129)

(133, 46) DF


,Unnamed: 0,ID,Programa,Tipo de Sondaje,NRO_SON,Sector,Este,Norte,Cota,Azimut,...,Q_INCLINA,Q_LARGO,Q_ESTE,Q_NORTE,Q_COTA,Q_ESTE_REC,Q_NORTE_REC,Q_COTA_REC,Q_SOPORTE,_merge
0,NaN,EVU_58,EVU,DDH-HQ3,DDH3958,Mirador Parapeto,359855.78,6491102.29,3560.35,270.0,...,-70.91,486.00,59855.78,91102.29,3560.35,59858,91097,3567,2.0,both
1,NaN,EVU_55,EVU,DDH-HQ3,DDH3959,Mirador Parapeto,359849.33,6491053.27,3560.77,270.0,...,-60.73,862.00,59849.33,91053.27,3560.77,59851,91050,3568,2.0,both
2,NaN,EVU_62,EVU,DDH-HQ3,DDH3961,Mirador Parapeto,359842.29,6491211.84,3552.90,267.0,...,-70.13,502.00,59842.29,91211.84,3552.9,59844,91207,3560,2.0,both
3,NaN,EVU_103,EVU,DDH-HQ3,DDH3962,Mirador Parapeto,359802.17,6491741.71,3529.38,298.0,...,-66.25,294.00,59802.17,91741.71,3529.38,59800,91745,3534,2.0,both
4,NaN,EVU_54,EVU,DDH-HQ3,DDH3963,Mirador Parapeto,359886.78,6491006.91,3561.87,270.0,...,-69.19,422.65,59886.78,91006.91,3561.87,59889,91000,3570,2.0,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,NaN,REC_PZVZ_46,HIDROGEOLOGÍA,DDH-HQ3,DDH4170,Fase 9 Banqueta 3020,359586.00,6489989.00,3020.00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
125,NaN,EVU139,EVU,DDH-HQ3,DDH4172,F12 Co Amarillo N Banco 3755,358815.00,6492143.00,3755.00,333.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
126,NaN,REC26_20,GEOTÉCNICO,DDH-HQ3,DDH4175,F9W Banco 2810,359132.00,6490342.00,2870.00,269.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
127,NaN,EVU141,EVU,DDH-HQ3,DDH4174,F12 Sector Norte Banco 3574,359896.00,6491919.00,3574.00,281.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [33]:
# Sin exportaci?n intermedia: DF se revisa en memoria.

### Segunda uni?n: coordenadas normalizadas + avance le?do desde el Excel

In [34]:
print('**** DF_FINAL = DF + df3 ****')
DF_Final = pd.merge(DF, df3, on=['NRO_SON'], how='outer', indicator='_merge_df3')
print(DF.shape,'DF')
DF_Final.head(5)

**** DF_FINAL = DF + df3 ****
(133, 46) DF


,Unnamed: 0,ID,Programa,Tipo de Sondaje,NRO_SON,Sector,Este,Norte,Cota,Azimut,...,AV_M_FOTOGRAFIA,AV_M_CORTE,AV_MAPEO_DESDE,AV_MAPEO_HASTA,AV_PREPARACION_DESDE,AV_PREPARACION_HASTA,Unnamed: 30,Unnamed: 31,Unnamed: 32,_merge_df3
0,NaN,EVU_58,EVU,DDH-HQ3,DDH3958,Mirador Parapeto,359855.78,6491102.29,3560.35,270.0,...,486.00,486.00,NaN,486.00,NaN,486.00,NaN,Etiquetas de fila,Suma de Tricono,both
1,NaN,EVU_55,EVU,DDH-HQ3,DDH3959,Mirador Parapeto,359849.33,6491053.27,3560.77,270.0,...,862.00,862.00,NaN,862.00,NaN,862.00,NaN,CATEGORIZACIÓN,NaN,both
2,NaN,EVU_62,EVU,DDH-HQ3,DDH3961,Mirador Parapeto,359842.29,6491211.84,3552.90,267.0,...,502.00,502.00,NaN,502.00,NaN,502.00,NaN,EVU,212.0,both
3,NaN,EVU_103,EVU,DDH-HQ3,DDH3962,Mirador Parapeto,359802.17,6491741.71,3529.38,298.0,...,294.00,294.00,NaN,294.00,NaN,294.00,NaN,GEOMETALÚRGICO,NaN,both
4,NaN,EVU_54,EVU,DDH-HQ3,DDH3963,Mirador Parapeto,359886.78,6491006.91,3561.87,270.0,...,422.65,422.65,NaN,422.65,NaN,422.65,NaN,GEOTÉCNICO,NaN,both


### Revisi?n del resultado de la segunda uni?n

Al igual que en la primera uni?n, se revisa el indicador antes de continuar. Se conservan `both` y `left_only` para que el universo final siga siendo el consolidado original. Los registros `right_only` corresponden a avances sin un sondaje presente en `CONSOLIDADO_PROGRAMA` y se muestran antes de excluirlos.

In [35]:
print(DF_Final['_merge_df3'].value_counts())
print('Avances sin registro en CONSOLIDADO_PROGRAMA:')
display(DF_Final.loc[DF_Final['_merge_df3'] == 'right_only', ['NRO_SON']])
DF_Final = DF_Final[DF_Final['_merge_df3'].isin(['both', 'left_only'])].copy()
print('Filas conservadas:', len(DF_Final))

_merge_df3
both          123
left_only      10
right_only      1
Name: count, dtype: int64
Avances sin registro en CONSOLIDADO_PROGRAMA:


,NRO_SON
133,123


Filas conservadas: 133


### RENOMBRO CAMPOS

### Homologaci?n al esquema de Collar_Recomendado

In [36]:
DF_Final= DF_Final.rename(columns={
    'ID':'R_NOMB_RECOM',
    'Tipo de Sondaje':'R_TIPOSONDAJE',
    'NRO_SON':'R_NRO_SON',
    'Sector':'R_SECTOR',
    'Este':'R_ESTE',
    'Norte':'R_NORTE',
    'Cota':'R_COTA',
    'Azimut':'R_AZIMUT',
    'Inclinación':'R_INCLINACION',
    'Largo (m)':'R_LARGO',
    'Fecha Inicio':'R_FCH INICIO',
    'Fecha Termino':'R_FCH TERMINO',
    'Por Perforar (m)':'R_POR_PERFORAR',
    'Avance Actual (m)':'R_AVANCE_ACTUAL',
    'Mts. Faltantes':'R_MTS_FALTANTES',
    '%Avance':'R_AVANCE',
    'Estatus Perforación (m)':'R_ESTATUS_PERF',
    'Largo Final (m)':'R_LARGO_FINAL',
    'Certificado Collar':'R_CERT_COLLAR',
    'Observación':'R_OBSERVACION',
    'Q_DES_CAMPANA':'Q_DES_CAMPAÑA',
    'Q_ANNO_SONDAJE':'Q_AÑO_SONDAJE',
    'Q_DES_TIPO_PERF':'Q_TIPO_PERF',
    'Q_DES_ESTADO_SON':'Q_ESTADO_SON',
    'AV_PROGRAMA':'AV_PROGRAMA',
    'AV_SONDA':'AV_SONDA',
    'AV_FCH_INI':'AV_FCH INI',
    'AV_FCH_TERM':'AV_FCH TERM',
    'AV_LARGO_PROGRAM':'AV_LARGO_PROGRAM',
    'AV_FONDO_FINAL':'AV_FONDO_FINAL',
    'AV_FALTANTE_PERF':'AV_FALTANTE_PERF',
    'AV_PCT_PERFORADO':'AV_PCT_PERFORADO',
    'AV_TRICONO':'AV_TRICONO',
    'AV_M_FOTOGRAFIA':'AV_MTS_FOTOGRAFIA',
    'AV_M_CORTE':'AV_MTS_CORTE',
    'AV_MAPEO_HASTA':'AV_MTS_MAPEO',
    'AV_PREPARACION_HASTA':'AV_MTS_PREPARACION'

})

In [37]:
DF_Final['AV_LARGO_PROGRAM'] = pd.to_numeric(DF_Final['AV_LARGO_PROGRAM'], errors='coerce')
DF_Final['AV_FONDO_FINAL'] = pd.to_numeric(DF_Final['AV_FONDO_FINAL'], errors='coerce')
DF_Final['AV_FALTANTE_PERF'] = pd.to_numeric(DF_Final['AV_FALTANTE_PERF'], errors='coerce')
DF_Final['AV_PCT_PERFORADO'] = pd.to_numeric(DF_Final['AV_PCT_PERFORADO'], errors='coerce')
DF_Final['AV_TRICONO'] = pd.to_numeric(DF_Final['AV_TRICONO'], errors='coerce')
DF_Final['AV_MTS_FOTOGRAFIA'] = pd.to_numeric(DF_Final['AV_MTS_FOTOGRAFIA'], errors='coerce')
DF_Final['AV_MTS_CORTE'] = pd.to_numeric(DF_Final['AV_MTS_CORTE'], errors='coerce')
DF_Final['AV_MTS_MAPEO'] = pd.to_numeric(DF_Final['AV_MTS_MAPEO'], errors='coerce')
DF_Final['AV_MTS_PREPARACION'] = pd.to_numeric(DF_Final['AV_MTS_PREPARACION'], errors='coerce')


In [38]:
DF_Final['R_FCH INICIO'] = pd.to_datetime(DF_Final['R_FCH INICIO'], format='%d/%m/%Y', errors='coerce')

### Normalizaci?n final de `Q_TIPO_PERF`

Los nulos, vac?os o espacios se reemplazan por **Sin datos**. Se muestran los casos para que el cliente contin?e incorporando y revisando reglas en este notebook.

In [39]:
tipo_perf_original = DF_Final['Q_TIPO_PERF'].copy()
DF_Final['Q_TIPO_PERF'] = DF_Final['Q_TIPO_PERF'].astype('string').str.strip().replace('', pd.NA).fillna('Sin datos')
mask_tipo_perf_normalizado = tipo_perf_original.isna() | tipo_perf_original.astype('string').str.strip().eq('')
print('Q_TIPO_PERF normalizados como Sin datos:', mask_tipo_perf_normalizado.sum())
DF_Final.loc[mask_tipo_perf_normalizado, ['R_NRO_SON', 'Q_TIPO_PERF']]

Q_TIPO_PERF normalizados como Sin datos: 21


,R_NRO_SON,Q_TIPO_PERF
22,DDH4022,Sin datos
36,RCH201,Sin datos
57,EVU87,Sin datos
61,RCH202,Sin datos
69,RCH204,Sin datos
71,EVU05_B,Sin datos
78,RCH205,Sin datos
89,RCH206,Sin datos
112,REC26_10,Sin datos
115,REC26_16,Sin datos


In [40]:
#print('**** DF_FINAL = DF + df3 ****')

#DF_FINAL = pd.merge(
 #   DF,
 #   df3,
 #   on='NRO_SON',
 #   how='outer',
 #   indicator='_merge_df3'
#)

#print(DF_FINAL.shape, 'DF_FINAL')
#DF_FINAL.head(68)

##### Creo nuevo DF1 SOLO CON COLUMNAS A EXPORTAR

In [41]:
DF1 = DF_Final[['R_NOMB_RECOM',	'R_TIPOSONDAJE',	'R_NRO_SON',	'R_SECTOR',	'R_ESTE',	'R_NORTE',	'R_COTA',	'R_AZIMUT',	'R_INCLINACION',	'R_LARGO',	'R_FCH INICIO',	'R_FCH TERMINO',	'R_POR_PERFORAR',	'R_AVANCE_ACTUAL',	'R_MTS_FALTANTES',	'R_AVANCE',	'R_ESTATUS_PERF',	'R_LARGO_FINAL',	'R_CERT_COLLAR',	'R_OBSERVACION',	'Q_DES_CAMPAÑA',	'Q_AÑO_SONDAJE',	'Q_TIPO_PERF',	'Q_ESTADO_SON',	'AV_PROGRAMA',	'AV_SONDA',	'AV_FCH INI',	'AV_FCH TERM',	'AV_LARGO_PROGRAM',	'AV_FONDO_FINAL',	'AV_FALTANTE_PERF',	'AV_PCT_PERFORADO',	'AV_TRICONO',	'AV_MTS_FOTOGRAFIA',	'AV_MTS_CORTE',	'AV_MTS_MAPEO',	'AV_MTS_PREPARACION',]]
DF1.head(5)

,R_NOMB_RECOM,R_TIPOSONDAJE,R_NRO_SON,R_SECTOR,R_ESTE,R_NORTE,R_COTA,R_AZIMUT,R_INCLINACION,R_LARGO,...,AV_FCH TERM,AV_LARGO_PROGRAM,AV_FONDO_FINAL,AV_FALTANTE_PERF,AV_PCT_PERFORADO,AV_TRICONO,AV_MTS_FOTOGRAFIA,AV_MTS_CORTE,AV_MTS_MAPEO,AV_MTS_PREPARACION
0,EVU_58,DDH-HQ3,DDH3958,Mirador Parapeto,359855.78,6491102.29,3560.35,270.0,-71.0,486.0,...,2024-09-23,486.0,486.00,0.00,100.000000,NaN,486.00,486.00,486.00,486.00
1,EVU_55,DDH-HQ3,DDH3959,Mirador Parapeto,359849.33,6491053.27,3560.77,270.0,-61.0,862.0,...,2024-10-12,862.0,862.00,0.00,100.000000,NaN,862.00,862.00,862.00,862.00
2,EVU_62,DDH-HQ3,DDH3961,Mirador Parapeto,359842.29,6491211.84,3552.90,267.0,-71.0,502.0,...,2024-09-29,502.0,502.00,0.00,100.000000,NaN,502.00,502.00,502.00,502.00
3,EVU_103,DDH-HQ3,DDH3962,Mirador Parapeto,359802.17,6491741.71,3529.38,298.0,-66.0,294.0,...,2024-10-03,294.0,294.00,0.00,100.000000,NaN,294.00,294.00,294.00,294.00
4,EVU_54,DDH-HQ3,DDH3963,Mirador Parapeto,359886.78,6491006.91,3561.87,270.0,-69.0,490.0,...,2024-10-18,490.0,422.65,67.35,86.255102,NaN,422.65,422.65,422.65,422.65


#### Resultado final

`DF1` mantiene las columnas finales y queda disponible para nuevas revisiones.

In [42]:
#DF1.columns = ['R_NOMB_RECOM','R_TIPOSONDAJE','R_NRO_SON','R_SECTOR',	'R_ESTE',	'R_NORTE',	'R_COTA',	'R_AZIMUT',	'R_INCLINACION',	'R_LARGO',	'R_FCH INICIO',	'R_FCH TERMINO',	'R_POR_PERFORAR',	'R_AVANCE_ACTUAL',	'R_MTS_FALTANTES',	'R_AVANCE',	'R_ESTATUS_PERF',	'R_LARGO_FINAL',	'R_CERT_COLLAR',	'R_OBSERVACION',	'Q_DES_CAMPAÑA',	'Q_AÑO_SONDAJE',	'Q_TIPO_PERF',	'Q_ESTADO_SON',	'AV_PROGRAMA',	'AV_SONDA',	'AV_FCH INI',	'AV_FCH TERM',	'AV_LARGO_PROGRAM',	'AV_FONDO_FINAL',	'AV_FALTANTE_PERF',	'AV_PCT_PERFORADO',	'AV_TRICONO',	'AV_MTS_FOTOGRAFIA',	'AV_MTS_CORTE',	'AV_MTS_MAPEO',	'AV_MTS_PREPARACION',]
#DF1.head(5)

#### Segunda unión DF1 con df3  (AREA_GRUPOS.csv)

In [43]:
#DF2 = pd.merge(DF1, df2, on=['GRUPO'])
#print('**** DF = Union USUARIOS_AMSA.csv USUARIOS_AMSA_AREA.csv ****')
#print('=============================================================')
#print(DF2.shape,'DF2')
#print(DF2.columns.tolist())
#DF2.head(5)


In [44]:
#DF1.to_excel("USUARIOS_AMSA_Jupyter5.xlsx",sheet_name="Matriz",index=False)

In [45]:
#df1.to_excel("USUARIOS_AMSA_original1.xlsx",sheet_name="Matriz",index=False)

In [46]:
print('Registros finales:', len(DF1))
print('Columnas finales:', len(DF1.columns))
display(DF1.head(20))
if EXPORTAR_RESULTADO:
    DF1.to_excel(ARCHIVO_RESULTADO, sheet_name='Matriz', index=False)
    print('Resultado exportado:', ARCHIVO_RESULTADO)
else:
    print('Exportaci?n desactivada; el resultado queda en DF1.')

Registros finales: 133
Columnas finales: 37


,R_NOMB_RECOM,R_TIPOSONDAJE,R_NRO_SON,R_SECTOR,R_ESTE,R_NORTE,R_COTA,R_AZIMUT,R_INCLINACION,R_LARGO,...,AV_FCH TERM,AV_LARGO_PROGRAM,AV_FONDO_FINAL,AV_FALTANTE_PERF,AV_PCT_PERFORADO,AV_TRICONO,AV_MTS_FOTOGRAFIA,AV_MTS_CORTE,AV_MTS_MAPEO,AV_MTS_PREPARACION
0,EVU_58,DDH-HQ3,DDH3958,Mirador Parapeto,359855.78,6491102.29,3560.35,270.0,-71.0,486.0,...,2024-09-23,486.0,486.00,0.00,100.000000,NaN,486.00,486.00,486.00,486.00
1,EVU_55,DDH-HQ3,DDH3959,Mirador Parapeto,359849.33,6491053.27,3560.77,270.0,-61.0,862.0,...,2024-10-12,862.0,862.00,0.00,100.000000,NaN,862.00,862.00,862.00,862.00
2,EVU_62,DDH-HQ3,DDH3961,Mirador Parapeto,359842.29,6491211.84,3552.90,267.0,-71.0,502.0,...,2024-09-29,502.0,502.00,0.00,100.000000,NaN,502.00,502.00,502.00,502.00
3,EVU_103,DDH-HQ3,DDH3962,Mirador Parapeto,359802.17,6491741.71,3529.38,298.0,-66.0,294.0,...,2024-10-03,294.0,294.00,0.00,100.000000,NaN,294.00,294.00,294.00,294.00
4,EVU_54,DDH-HQ3,DDH3963,Mirador Parapeto,359886.78,6491006.91,3561.87,270.0,-69.0,490.0,...,2024-10-18,490.0,422.65,67.35,86.255102,NaN,422.65,422.65,422.65,422.65
5,EVU_63,DDH-HQ3,DDH3964,Mirador F9,359842.10,6491212.82,3552.68,276.0,-61.0,878.0,...,2024-10-27,878.0,878.00,0.00,100.000000,NaN,878.00,878.00,878.00,878.00
6,EVU_81,DDH-HQ3,DDH3966,Mirador Parapeto,359804.09,6491743.32,3529.42,260.0,-58.0,516.0,...,2024-10-27,516.0,516.00,0.00,100.000000,NaN,516.00,516.00,516.00,516.00
7,EVU_47,DDH-HQ3,DDH3969,F9 Parapeto,359941.96,6490810.54,3582.55,269.0,-59.0,823.0,...,2024-11-17,823.0,823.00,0.00,100.000000,NaN,823.00,823.00,823.00,823.00
8,EVU_51,DDH-HQ3,DDH3972,F9 Parapeto,359899.71,6490922.27,3567.78,262.0,-60.0,776.0,...,2024-12-05,776.0,776.00,0.00,100.000000,NaN,776.00,776.00,776.00,776.00
9,EVU_59_,DDH-HQ3,DDH3968A,Mirador Parapeto,359856.43,6491149.65,3557.31,270.0,-60.0,880.0,...,2024-11-14,880.0,880.00,0.00,100.000000,NaN,880.00,880.00,880.00,880.00


Exportaci?n desactivada; el resultado queda en DF1.
